# Mélanger des lumières

| Module | Durée | Prérequis |
|---|---|---|
| M1 – Vecteurs et ressemblance | 75 min | M0 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une petite salle de théâtre dispose de **trois projecteurs colorés** : un rouge orangé, un vert et un bleu. Chacun a un variateur, réglable de 0 % (éteint) à 100 % (pleine puissance). La metteuse en scène demande quatre ambiances :

1. un **blanc chaud** pour la scène d'ouverture ;
2. un **orange** de coucher de soleil ;
3. un **jaune vif** pour la scène du marché ;
4. un **blanc intense**, très lumineux, pour le final.

**La question du régisseur : pour chaque ambiance, comment régler les trois variateurs ? Et sont-elles toutes réalisables ?**

**Activité** : De quelles informations a-t-on besoin sur les projecteurs ? Sur les couleurs demandées ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_realisables = 4      # combien des 4 ambiances sont réalisables ? (modifiez)

## ③ Abstraire

**Coder une couleur.** Un écran, une caméra ou un projecteur décrivent une couleur par trois nombres entre 0 et 1 : son intensité de **rouge, vert et bleu** (R, V, B). Une couleur est donc un **vecteur de dimension 3**.

Un technicien a mesuré la couleur de chaque projecteur à pleine puissance, et la metteuse en scène a choisi ses couleurs sur un nuancier :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

projecteurs = {
    "rouge orangé": np.array([1.00, 0.35, 0.10]),
    "vert":         np.array([0.15, 0.90, 0.25]),
    "bleu":         np.array([0.10, 0.20, 0.95]),
}
cibles = {
    "blanc chaud":   np.array([0.95, 0.85, 0.60]),
    "orange":        np.array([0.85, 0.55, 0.15]),
    "jaune vif":     np.array([0.95, 0.95, 0.05]),
    "blanc intense": np.array([1.30, 1.40, 1.30]),   # plus de 1 : plus lumineux qu'un seul projecteur
}

def cadre(ax):
    # Pastille encadrée de gris, pour qu'un blanc reste visible sur fond blanc
    ax.set_xticks([]); ax.set_yticks([])
    for bord in ax.spines.values():
        bord.set_color("0.6")

def nuancier(couleurs, titre):
    fig, axes = plt.subplots(1, len(couleurs), figsize=(2.2 * len(couleurs), 2))
    for ax, (nom, c) in zip(np.atleast_1d(axes), couleurs.items()):
        ax.imshow([[np.clip(c, 0, 1)]]); ax.set_title(nom, fontsize=10); cadre(ax)
    fig.suptitle(titre); plt.show()

nuancier(projecteurs, "Les projecteurs à pleine puissance")
nuancier(cibles, "Les ambiances demandées")

**Le concept : la combinaison linéaire.** La lumière **s'additionne** : deux projecteurs braqués au même endroit donnent la somme de leurs couleurs. Et un variateur réglé à 40 % multiplie la couleur par 0,4. Si les variateurs sont réglés à $a$, $b$ et $c$ (entre 0 et 1), la couleur obtenue est

$$
a \cdot \text{rouge} + b \cdot \text{vert} + c \cdot \text{bleu}
$$

On appelle cela une **combinaison linéaire** des trois vecteurs. La question devient : trouver $a$, $b$, $c$ tels que cette combinaison soit égale à la couleur cible, avec

$$
0 \le a \le 1, \qquad 0 \le b \le 1, \qquad 0 \le c \le 1.
$$

**Le concept s'applique-t-il ?.** Pourquoi parle-t-on de lumières et pas de peintures ? Mélanger des peintures ne revient pas à additionner leurs couleurs : chaque pigment **absorbe** une partie de la lumière, et un mélange de peintures est plus sombre que chacune d'elles. Le modèle linéaire décrit bien les lumières ; pour les peintures, il ne serait qu'une approximation grossière.

**Activité** : Quelle couleur obtient-on avec le rouge orangé à 50 % et le vert à 50 %, le bleu éteint ?

Indiquez vos réponses et justifiez-les.

In [ ]:
melange = 0.5 * projecteurs["rouge orangé"] + 0.5 * projecteurs["vert"]
print(melange)
nuancier({"50 % rouge + 50 % vert": melange}, "Vérification")

## ④ Calculer

Trouver a, b, c revient à résoudre 3 équations (une par composante R, V, B) à 3 inconnues. Vous apprendrez à le faire en détail au module M2. Ici, utilisez l'outil `np.linalg.solve` **comme une boîte noire**, en appliquant la méthode du module M0 : lisez sa documentation, puis testez-le.

Il attend une matrice dont les **colonnes** sont les couleurs des projecteurs, et le vecteur cible.

In [ ]:
A = np.column_stack(list(projecteurs.values()))   # colonnes : rouge orangé, vert, bleu
print(A)

# Test sur un cas dont on connaît la réponse : 50 % rouge, 50 % vert, 0 % bleu
print("Réglages retrouvés :", np.linalg.solve(A, melange).round(6))

In [ ]:
reglages = {}
for nom, cible in cibles.items():
    x = np.linalg.solve(A, cible)
    reglages[nom] = x
    realisable = np.all((x >= 0) & (x <= 1))
    print(f"{nom:14s} réglages (a, b, c) = {x.round(3)}  ->  {'réalisable' if realisable else 'IMPOSSIBLE'}")

## ⑤ Interpréter

**Activité** : Pour chaque ambiance impossible, que signifie concrètement le réglage obtenu ?

- Un réglage **négatif** : il faudrait « retirer » de la lumière d'un projecteur. Est-ce possible ?
- Un réglage **supérieur à 1** : il faudrait un projecteur plus puissant que le nôtre. Que proposer au régisseur ?
- Le cas de l'**orange** est limite : le réglage du bleu vaut environ −0,006. Faut-il vraiment dire « impossible » ?

Indiquez vos réponses et justifiez-les.

In [ ]:
# Ce que le régisseur peut faire de mieux : ramener chaque réglage dans [0, 1]
approx = {nom: A @ np.clip(x, 0, 1) for nom, x in reglages.items()}
fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for j, nom in enumerate(cibles):
    axes[0, j].imshow([[np.clip(cibles[nom], 0, 1)]]); axes[0, j].set_title(nom, fontsize=10)
    axes[1, j].imshow([[np.clip(approx[nom], 0, 1)]])
    cadre(axes[0, j]); cadre(axes[1, j])
axes[0, 0].set_ylabel("demandé"); axes[1, 0].set_ylabel("obtenu")
plt.suptitle("Couleur demandée et meilleure approximation simple"); plt.show()

nb_ok = sum(np.all((x >= -1e-2) & (x <= 1)) for x in reglages.values())
print(f"Votre pari : {pari_realisables} ambiances réalisables ; avec une tolérance de 1 % : {nb_ok}")

### Ce que peuvent produire deux projecteurs, puis trois

Choisissons au hasard des milliers de réglages et plaçons chaque couleur obtenue dans l'espace (R, V, B).

In [ ]:
rng = np.random.default_rng(0)
fig = plt.figure(figsize=(10, 4.5))
for k, (titre, nb_proj) in enumerate([("Deux projecteurs (rouge, vert)", 2), ("Trois projecteurs", 3)]):
    poids = rng.random((3000, nb_proj))
    couleurs = poids @ A[:, :nb_proj].T
    ax = fig.add_subplot(1, 2, k + 1, projection="3d")
    ax.scatter(*couleurs.T, c=np.clip(couleurs, 0, 1), s=4)
    ax.set_xlabel("R"); ax.set_ylabel("V"); ax.set_zlabel("B"); ax.set_title(titre)
plt.show()

**Activité** :

1. Avec deux projecteurs, les couleurs obtenues forment-elles un volume, une surface ou une ligne ? Pourquoi ?
2. Avec trois projecteurs, quelle forme obtient-on ? Où se trouve le « jaune vif » par rapport à cette forme ?
3. **Même structure, autre situation.** Une diététicienne veut composer un repas à partir de trois aliments pour atteindre exactement une cible en protéines, glucides et lipides. Identifiez les vecteurs, les coefficients et les contraintes. Qu'est-ce qui est identique au problème des projecteurs ? Qu'est-ce qui diffère ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** On a supposé qu'un variateur à 40 % donne exactement 40 % de la couleur. Avec de vraies lampes, ce n'est souvent pas le cas. Quelles conclusions resteraient vraies si ce n'était qu'approximatif ?
- **Généraliser.** Écrivez une fonction `reglage(projecteurs, cible, tolerance)` qui renvoie les réglages, ou `None` si l'ambiance est impossible. Ajoutez un quatrième projecteur jaune : pourquoi `np.linalg.solve` refuse-t-il alors de calculer ? (Vous y reviendrez au module M2.)

In [ ]:
def reglage(projecteurs, cible, tolerance=0.01):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def reglage(projecteurs, cible, tolerance=0.01):
    A = np.column_stack(list(projecteurs.values()))
    x = np.linalg.solve(A, cible)
    if np.all((x >= -tolerance) & (x <= 1 + tolerance)):
        return np.clip(x, 0, 1)
    return None

for nom, cible in cibles.items():
    print(nom, reglage(projecteurs, cible))

avec_jaune = {**projecteurs, "jaune": np.array([0.95, 0.90, 0.10])}
try:
    reglage(avec_jaune, cibles["orange"])
except np.linalg.LinAlgError as err:
    print("Quatre projecteurs :", err, "— la matrice a 3 lignes et 4 colonnes, elle n'est pas carrée.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Identifier les concepts pertinents et leurs relations": 1,
    "Reconnaître si le concept s'applique": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Repérer ressemblances et différences entre situations pour abstraire": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "1.2-melanger-lumieres", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")